# Load modules

In [2]:
import pymysql
import pandas as pd

In [13]:
import requests

# Read in the mysql job in the node it is being run

In [3]:
conn = pymysql.connect(host='node-13-09', port=3307, user='root', password='', database='merops')

for t in ['code', 'domain', 'sequence', 'organism', 'gene_name', 'cleavage', 'substrate']:
    n = pd.read_sql(f"SELECT COUNT(*) as n FROM {t}", conn).iloc[0,0]
    print(f"{t}: {n} rows")

/tmp/ipykernel_631105/1896644190.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  n = pd.read_sql(f"SELECT COUNT(*) as n FROM {t}", conn).iloc[0,0]


code: 6949 rows
domain: 1327336 rows
sequence: 1256053 rows
organism: 32241 rows
gene_name: 936026 rows
cleavage: 179080 rows
substrate: 77448 rows


## Subset human data

In [4]:
human_organism = pd.read_sql("SELECT * FROM organism WHERE taxonomy_id = 9606", conn)
print(human_organism)

  merops_taxonomy_id  taxonomy_id complete_genome  gene_total  \
0           sp001823         9606             Yes       54136   

   peptidase_count  peptidase_homologue_count  inhibitor_count  \
0             1252                        349             1702   

   inhibitor_homologue_count   kingdom  
0                         94  Animalia  


/tmp/ipykernel_631105/3922585198.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  human_organism = pd.read_sql("SELECT * FROM organism WHERE taxonomy_id = 9606", conn)


### check

In [5]:
merops_codes = pd.read_sql("SELECT * FROM code WHERE family IN ('M12', 'M10')", conn)
print(merops_codes.shape)
print(merops_codes[['code','family','subfamily']].head(30))

(345, 10)
       code family subfamily
0   M10.001    M10      M10A
1   M10.002    M10      M10A
2   M10.003    M10      M10A
3   M10.004    M10      M10A
4   M10.005    M10      M10A
5   M10.006    M10      M10A
6   M10.007    M10      M10A
7   M10.008    M10      M10A
8   M10.009    M10      M10A
9   M10.010    M10      M10A
10  M10.011    M10      M10A
11  M10.012    M10      M10A
12  M10.013    M10      M10A
13  M10.014    M10      M10A
14  M10.015    M10      M10A
15  M10.016    M10      M10A
16  M10.017    M10      M10A
17  M10.018    M10      M10A
18  M10.019    M10      M10A
19  M10.020    M10      M10C
20  M10.021    M10      M10A
21  M10.022    M10      M10A
22  M10.023    M10      M10A
23  M10.024    M10      M10A
24  M10.025    M10      M10A
25  M10.026    M10      M10A
26  M10.027    M10      M10A
27  M10.029    M10      M10A
28  M10.030    M10      M10A
29  M10.031    M10      M10A


/tmp/ipykernel_631105/266130143.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  merops_codes = pd.read_sql("SELECT * FROM code WHERE family IN ('M12', 'M10')", conn)


In [6]:
query = """
SELECT DISTINCT c.code, c.family, c.subfamily, gn.gene_name
FROM code c
JOIN domain d ON c.code = d.code
JOIN sequence s ON d.sequence_id = s.sequence_id
JOIN gene_name gn ON s.sequence_id = gn.sequence_id
WHERE s.merops_taxonomy_id = 'sp001823'
AND (gn.gene_name LIKE 'ADAM%' OR gn.gene_name LIKE 'MMP%')
ORDER BY gn.gene_name
"""
sheddase_codes = pd.read_sql(query, conn)
print(sheddase_codes.shape)
sheddase_codes

(67, 4)


/tmp/ipykernel_631105/916298300.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  sheddase_codes = pd.read_sql(query, conn)


,code,family,subfamily,gene_name
0,M12.P03,M12,M12B,ADAM1
1,M12.210,M12,M12B,ADAM10
2,M12.976,M12,M12B,ADAM11
3,M12.212,M12,M12B,ADAM12
4,M12.215,M12,M12B,ADAM15
...,...,...,...,...
62,M10.005,M10,M10A,MMP3
63,M10.008,M10,M10A,MMP7
64,M10.002,M10,M10A,MMP8
65,M10.004,M10,M10A,MMP9


In [7]:
sheddase_codes_clean = sheddase_codes[~sheddase_codes.gene_name.str.contains('withdrawn')]
print(sheddase_codes_clean.shape)
print(sorted(sheddase_codes_clean.gene_name.unique()))

(66, 4)
['ADAM1', 'ADAM10', 'ADAM11', 'ADAM12', 'ADAM15', 'ADAM17', 'ADAM18', 'ADAM19', 'ADAM2', 'ADAM20', 'ADAM21', 'ADAM22', 'ADAM23', 'ADAM28', 'ADAM29', 'ADAM30', 'ADAM32', 'ADAM33', 'ADAM3A', 'ADAM3B', 'ADAM7', 'ADAM8', 'ADAM9', 'ADAMDEC1', 'ADAMTS1', 'ADAMTS10', 'ADAMTS12', 'ADAMTS13', 'ADAMTS14', 'ADAMTS15', 'ADAMTS16', 'ADAMTS17', 'ADAMTS18', 'ADAMTS19', 'ADAMTS2', 'ADAMTS20', 'ADAMTS3', 'ADAMTS4', 'ADAMTS5', 'ADAMTS6', 'ADAMTS7', 'ADAMTS8', 'ADAMTS9', 'MMP1', 'MMP10', 'MMP11', 'MMP12', 'MMP13', 'MMP14', 'MMP15', 'MMP16', 'MMP17', 'MMP19', 'MMP2', 'MMP20', 'MMP21', 'MMP23B', 'MMP24', 'MMP25', 'MMP26', 'MMP27', 'MMP28', 'MMP3', 'MMP7', 'MMP8', 'MMP9']


## extract all peptidase - protein substrate subsets

In [9]:
query = """
SELECT DISTINCT c.code, gn.gene_name AS peptidase_gene, cl.uniprot_acc AS substrate_uniprot,
       cl.cleavage_evidence, cl.cleavage_type
FROM code c
JOIN domain d ON c.code = d.code
JOIN sequence s ON d.sequence_id = s.sequence_id
JOIN gene_name gn ON s.sequence_id = gn.sequence_id
JOIN cleavage cl ON c.code = cl.code
JOIN substrate sub ON cl.uniprot_acc = sub.uniprot_acc
WHERE s.merops_taxonomy_id = 'sp001823'
AND sub.taxonomy_id = 9606
AND gn.type = 'name'
"""
peptidase_substrate_human = pd.read_sql(query, conn)
print(peptidase_substrate_human.shape)
peptidase_substrate_human.head(10)

(12649, 5)


/tmp/ipykernel_631105/1283309899.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  peptidase_substrate_human = pd.read_sql(query, conn)


,code,peptidase_gene,substrate_uniprot,cleavage_evidence,cleavage_type
0,M67.006,STAMBP,P0CG47,experimental,non-physiological
1,C14.018,CASP14,O75822,experimental,physiological
2,C14.018,CASP14,P08238,experimental,physiological
3,C14.018,CASP14,P11387,experimental,physiological
4,C14.018,CASP14,P14625,experimental,physiological
5,C14.018,CASP14,P23588,experimental,physiological
6,C14.018,CASP14,P60896,experimental,physiological
7,C14.018,CASP14,Q32MZ4,experimental,physiological
8,C14.018,CASP14,Q58FF7,experimental,physiological
9,C14.018,CASP14,Q9NR30,experimental,physiological


### Mapping Substrate UniProt accession to ENSG

In [10]:
gtf_path = '/nfs/users/nfs_m/mt19/cardinal_analysis/ht/datasets/Flanders/trans_eQTLS_ERs/Homo_sapiens.GRCh38.110.gtf.gz'
gtf_cols = ['seqname','source','feature','start','end','score','strand','frame','attribute']
gtf = pd.read_csv(gtf_path, sep='\t', comment='#', names=gtf_cols, dtype={'seqname': str}, low_memory=False)
print(gtf.shape)
print(gtf.feature.value_counts())

(3421622, 9)
feature
exon               1649677
CDS                 885265
transcript          252894
three_prime_utr     207640
five_prime_utr      173466
start_codon          97935
stop_codon           91861
gene                 62754
Selenocysteine         130
Name: count, dtype: int64


In [11]:
gtf[gtf.feature == 'gene'].attribute.iloc[0]

'gene_id "ENSG00000279928"; gene_version "2"; gene_name "DDX11L17"; gene_source "havana"; gene_biotype "unprocessed_pseudogene";'

In [12]:
gtf[gtf.feature == 'CDS'].attribute.iloc[0]

'gene_id "ENSG00000142611"; gene_version "17"; transcript_id "ENST00000511072"; transcript_version "5"; exon_number "1"; gene_name "PRDM16"; gene_source "ensembl_havana"; gene_biotype "protein_coding"; transcript_name "PRDM16-206"; transcript_source "havana"; transcript_biotype "protein_coding"; protein_id "ENSP00000426975"; protein_version "1"; tag "basic"; transcript_support_level "5";'

In [35]:
genes = gtf[gtf.feature == 'gene'].copy()
genes['gene_id'] = genes.attribute.str.extract(r'gene_id "([^"]+)"')
genes['gene_name'] = genes.attribute.str.extract(r'gene_name "([^"]+)"')
symbol_to_ensembl = genes[['gene_name','gene_id']].dropna().drop_duplicates()
print(symbol_to_ensembl.shape)

(42361, 2)


### test API

In [14]:
r = requests.get('https://rest.uniprot.org/idmapping/run', timeout=10)
print(r.status_code)

500


In [23]:
has_uniprot = gtf.attribute.str.contains('P0CG47', case=False, na=False)
print(has_uniprot.sum())

0


In [15]:
import requests
import time

substrate_ids = peptidase_substrate_human.substrate_uniprot.unique().tolist()
print(len(substrate_ids))

r = requests.post(
    'https://rest.uniprot.org/idmapping/run',
    data={'from': 'UniProtKB_AC-ID', 'to': 'Ensembl', 'ids': ','.join(substrate_ids[:100])}
)
print(r.status_code, r.json())

3905
200 {'jobId': 'xeiHOqwbaK'}


In [16]:
job_id = 'xeiHOqwbaK'

while True:
    status = requests.get(f'https://rest.uniprot.org/idmapping/status/{job_id}')
    data = status.json()
    if 'results' in data or 'failedIds' in data:
        break
    print('still running...', data)
    time.sleep(3)

print(data.keys())
print(len(data.get('results', [])), 'mapped')
print(len(data.get('failedIds', [])), 'failed')

dict_keys(['results', 'failedIds'])
25 mapped
2 failed


In [17]:
print(len(substrate_ids[:100]))  # confirm we actually sent 100
print(data['results'][:5])
print(data['failedIds'][:5])

100
[{'from': 'P0CG47', 'to': 'ENSG00000170315.15'}, {'from': 'O75822', 'to': 'ENSG00000104131.14'}, {'from': 'P08238', 'to': 'ENSG00000096384.21'}, {'from': 'P11387', 'to': 'ENSG00000198900.7'}, {'from': 'P14625', 'to': 'ENSG00000166598.17'}]
['P00736', 'Q58FF7']


In [18]:
print(status.headers.get('Link'))
print(status.headers.get('x-total-results'))
print(status.url)

<https://rest.uniprot.org/idmapping/results/xeiHOqwbaK?cursor=3v5g94y9lqs&size=25>; rel="next"
113
https://rest.uniprot.org/idmapping/results/xeiHOqwbaK


In [19]:
r = requests.get(f'https://rest.uniprot.org/idmapping/stream/{job_id}?format=json')
data = r.json()
print(len(data.get('results', [])), 'mapped')
print(len(data.get('failedIds', [])), 'failed')

113 mapped
0 failed


In [24]:
r = requests.get(f'https://rest.uniprot.org/idmapping/stream/{job_id}?format=json')
data = r.json()
print("mapped:", len(data.get('results', [])), "failed:", len(data.get('failedIds', [])))

mapped: 113 failed: 0


In [25]:
unique_from = set(r['from'] for r in data['results'])
print("unique source UniProt IDs in results:", len(unique_from))
print("total result rows:", len(data['results']))

unique source UniProt IDs in results: 98
total result rows: 113


### Final query

In [26]:
r = requests.post(
    'https://rest.uniprot.org/idmapping/run',
    data={'from': 'UniProtKB_AC-ID', 'to': 'Ensembl', 'ids': ','.join(substrate_ids)}
)
print(r.status_code, r.json())

200 {'jobId': 'netJn1CjKF'}


In [27]:
full_job_id = r.json()['jobId']
while True:
    status = requests.get(f'https://rest.uniprot.org/idmapping/status/{full_job_id}')
    data = status.json()
    if 'results' in data or 'failedIds' in data or 'jobStatus' in data and data.get('jobStatus') == 'FINISHED':
        break
    print('still running...')
    time.sleep(5)
print('done, retrieving via stream endpoint...')

done, retrieving via stream endpoint...


In [28]:
r_stream = requests.get(f'https://rest.uniprot.org/idmapping/stream/{full_job_id}?format=json')
full_data = r_stream.json()

unique_mapped = set(item['from'] for item in full_data.get('results', []))
print("unique source UniProt IDs mapped:", len(unique_mapped))
print("total result rows:", len(full_data.get('results', [])))
print("failed IDs:", len(full_data.get('failedIds', [])))
print("mapped + failed vs submitted:", len(unique_mapped) + len(full_data.get('failedIds', [])), "vs", len(substrate_ids))

unique source UniProt IDs mapped: 3858
total result rows: 4325
failed IDs: 0
mapped + failed vs submitted: 3858 vs 3905


In [29]:
submitted_set = set(substrate_ids)
accounted = unique_mapped | set(full_data.get('failedIds', []))
missing = submitted_set - accounted
print(len(missing))
print(sorted(missing)[:20])

47
['A6NEC2', 'A6NIZ1', 'A6NKF1', 'A6NMY6', 'A8MUU1', 'O60361', 'O94854', 'P00736', 'P01860', 'P01893', 'P02746', 'P0C7P4', 'P10163', 'P23588', 'P26927', 'P48741', 'Q14397', 'Q14568', 'Q58FF6', 'Q58FF7']


In [30]:
# spot-check a few directly against UniProt's REST API to see their current status
for acc in sorted(missing)[:5]:
    resp = requests.get(f'https://rest.uniprot.org/uniprotkb/{acc}.json')
    print(acc, resp.status_code)

A6NEC2 200
A6NIZ1 200
A6NKF1 200
A6NMY6 200
A8MUU1 200


In [31]:
for acc in sorted(missing)[:10]:
    resp = requests.get(f'https://rest.uniprot.org/uniprotkb/{acc}.json')
    entry = resp.json()
    name = entry.get('proteinDescription', {}).get('recommendedName', {}).get('fullName', {}).get('value', '?')
    has_ensembl_xref = any(db.get('database') == 'Ensembl' for db in entry.get('uniProtKBCrossReferences', []))
    print(acc, '-', name, '- has Ensembl xref:', has_ensembl_xref)

A6NEC2 - Puromycin-sensitive aminopeptidase-like protein - has Ensembl xref: False
A6NIZ1 - Ras-related protein Rap-1b-like protein - has Ensembl xref: False
A6NKF1 - SAC3 domain-containing protein 1 - has Ensembl xref: False
A6NMY6 - Putative annexin A2-like protein - has Ensembl xref: False
A8MUU1 - Putative fatty acid-binding protein 5-like protein 3 - has Ensembl xref: False
O60361 - ? - has Ensembl xref: False
O94854 - Microtubule-actin cross-linking factor 1, isoforms 6/7 - has Ensembl xref: False
P00736 - Complement C1r subcomponent - has Ensembl xref: False
P01860 - Immunoglobulin heavy constant gamma 3 - has Ensembl xref: False
P01893 - Putative HLA class I histocompatibility antigen, alpha chain H - has Ensembl xref: False


In [32]:
uniprot2ensg = pd.DataFrame(full_data['results'])
uniprot2ensg['ensembl_gene'] = uniprot2ensg['to'].str.split('.').str[0]  # strip version suffix
uniprot2ensg = uniprot2ensg[['from', 'ensembl_gene']].rename(columns={'from': 'substrate_uniprot'}).drop_duplicates()
print(uniprot2ensg.shape)
uniprot2ensg.head()

(4325, 2)


,substrate_uniprot,ensembl_gene
0,P0CG47,ENSG00000170315
1,O75822,ENSG00000104131
2,P08238,ENSG00000096384
3,P11387,ENSG00000198900
4,P14625,ENSG00000166598


In [33]:
print(uniprot2ensg.substrate_uniprot.nunique())

3858


In [36]:
sym2ens_gtf = symbol_to_ensembl.groupby('gene_name')['gene_id'].apply(lambda s: sorted(set(s))).to_dict()

peptidase_substrate_human['peptidase_ensembl'] = peptidase_substrate_human.peptidase_gene.map(sym2ens_gtf)
n_unmapped_peptidase = peptidase_substrate_human.peptidase_ensembl.isna().sum()
print(f'peptidase rows unmapped: {n_unmapped_peptidase} / {len(peptidase_substrate_human)}')

peptidase rows unmapped: 875 / 12649


In [37]:
unmapped_genes = peptidase_substrate_human[peptidase_substrate_human.peptidase_ensembl.isna()].peptidase_gene.unique()
print(len(unmapped_genes))
print(sorted(unmapped_genes)[:30])

7
['C9orf3', 'CTSL1', 'EMR2', 'FAM105A', 'FAM105B', 'LPHN1', 'LPHN3']


In [38]:
old_to_new = {
    'CTSL1': 'CTSL', 'EMR2': 'ADGRE2', 'LPHN1': 'ADGRL1', 'LPHN3': 'ADGRL3',
    'FAM105A': 'OTULINL', 'FAM105B': 'OTULIN', 'C9orf3': 'AOPEP'
}
for old, new in old_to_new.items():
    print(old, '->', new, ':', new in sym2ens_gtf)

CTSL1 -> CTSL : True
EMR2 -> ADGRE2 : True
LPHN1 -> ADGRL1 : True
LPHN3 -> ADGRL3 : True
FAM105A -> OTULINL : True
FAM105B -> OTULIN : True
C9orf3 -> AOPEP : True


In [39]:
peptidase_substrate_human['peptidase_gene_mapped'] = peptidase_substrate_human.peptidase_gene.replace(old_to_new)
peptidase_substrate_human['peptidase_ensembl'] = peptidase_substrate_human.peptidase_gene_mapped.map(sym2ens_gtf)

n_unmapped = peptidase_substrate_human.peptidase_ensembl.isna().sum()
print(f'peptidase rows unmapped after rename fix: {n_unmapped} / {len(peptidase_substrate_human)}')

peptidase rows unmapped after rename fix: 0 / 12649


In [40]:
peptidase_substrate_human_exploded = peptidase_substrate_human.explode('peptidase_ensembl')

merops_ensembl_pairs = peptidase_substrate_human_exploded.merge(
    uniprot2ensg, on='substrate_uniprot', how='left'
)

n_no_substrate_map = merops_ensembl_pairs.ensembl_gene.isna().sum()
print(f'rows with no substrate Ensembl mapping (the 47 unmappable UniProt IDs): {n_no_substrate_map} / {len(merops_ensembl_pairs)}')

merops_ensembl_final = merops_ensembl_pairs.dropna(subset=['ensembl_gene']).copy()
merops_ensembl_final = merops_ensembl_final.rename(columns={'peptidase_ensembl': 'peptidase_ensembl_gene', 'ensembl_gene': 'substrate_ensembl_gene'})

print('final peptidase-substrate ENSG pairs:', merops_ensembl_final[['peptidase_ensembl_gene','substrate_ensembl_gene']].drop_duplicates().shape[0])
merops_ensembl_final[['code','peptidase_gene','peptidase_ensembl_gene','substrate_uniprot','substrate_ensembl_gene','cleavage_evidence','cleavage_type']].head()

rows with no substrate Ensembl mapping (the 47 unmappable UniProt IDs): 128 / 14414
final peptidase-substrate ENSG pairs: 13915


,code,peptidase_gene,peptidase_ensembl_gene,substrate_uniprot,substrate_ensembl_gene,cleavage_evidence,cleavage_type
0,M67.006,STAMBP,ENSG00000124356,P0CG47,ENSG00000170315,experimental,non-physiological
1,C14.018,CASP14,ENSG00000105141,O75822,ENSG00000104131,experimental,physiological
2,C14.018,CASP14,ENSG00000105141,P08238,ENSG00000096384,experimental,physiological
3,C14.018,CASP14,ENSG00000105141,P11387,ENSG00000198900,experimental,physiological
4,C14.018,CASP14,ENSG00000105141,P14625,ENSG00000166598,experimental,physiological


In [41]:
merops_ensembl_final.to_csv('/nfs/team151/mt19/merops_db/merops_peptidase_substrate_ensembl_human.csv', index=False)
print(merops_ensembl_final.shape)

(14286, 8)


In [42]:
print(merops_ensembl_final.cleavage_evidence.value_counts())

cleavage_evidence
experimental    14286
Name: count, dtype: int64


In [43]:
print(merops_ensembl_final.cleavage_type.value_counts())

cleavage_type
non-physiological    8044
physiological        5860
synthetic             382
Name: count, dtype: int64
